In [ ]:
import torch
import torch.nn
# from torchview import draw_graph

class Quantize(nn.Module):

    def __init__(self):
        super().__init__()
        self.enc = nn.Linear(1, 1)
        self.codebook = nn.Parameter(torch.tensor([-10., 0., 4., 3.]))
        self.dec = nn.Linear(1, 1)

    def forward(self, x):
        z = self.enc(x)
        closest = torch.argmin((z - self.codebook)**2, dim=-1)
        e_j = self.codebook[closest]
        e_j = z + (e_j - z).detach()
        d = self.dec(e_j)
        return d



In [ ]:
q = Quantize()
x = torch.ones(1).unsqueeze(0)
out = q(x)
out.backward()

# model_graph = draw_graph(q, input_size=(1, 1))
# model_graph.visual_graph

let's build our vqvae

In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import MNIST
from tqdm.auto import tqdm
from IPython.display import HTML
from sklearn.manifold import TSNE


In [ ]:
!pip install celluloid

### Stuff to Visualize the Latent Space ###
from celluloid import Camera
from IPython.display import HTML

In [ ]:

### Seed Everything ###
torch.manual_seed(0)
torch.cuda.manual_seed(0)
np.random.seed(0)
random.seed(0)


### Prep Dataset ###
tensor_transforms = transforms.Compose(
    [
        transforms.Resize((32,32)),
        transforms.ToTensor()
    ]
)
train_set = MNIST("../../../data/mnist/", download=True, train=True, transform=tensor_transforms)
test_set = MNIST("../../../data/mnist/", download=True, train=False, transform=tensor_transforms)


### Set Device ###
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:


class VectorQuantizer(nn.Module):

    def __init__(self, codebook_size=1024, latent_dim=2):
        super().__init__()
        self.embedding = nn.Embedding(codebook_size, latent_dim)
        self.embedding.weight.data.uniform_(-1/codebook_size, 1/codebook_size)

        self.latent_dim = latent_dim
        self.codebook_size = codebook_size


    def forward(self, x, efficient = False):

        batch_size = x.shape[0]

        if not efficient:

            # Embed: [B x C x L]
            # Data: [B x 1 x L]
            emb = self.embedding.weight.unsqueeze(0).repeat(batch_size, 1, 1)
            x = x.unsqueeze(1)
            distances = torch.sum((x - emb)**2, dim=-1)


        else:

            # (L-C)**2 = (L2 - 2LC + C2)
            L2 = torch.sum(x**2, dim=-1, keepdim=True)
            C2 = torch.sum(self.embedding.weight**2, dim=-1).unsqueeze(0)
            LC = x @ self.embedding.weight.t()
            distances = L2 - 2*LC + C2

        closest = torch.argmin(distances, dim=-1)

        quantized_latents_idx = torch.zeros(batch_size, self.codebook_size, device=x.device)

        batch_idx = torch.arange(batch_size)
        quantized_latents_idx[batch_idx, closest] = 1

        quantized_latents = quantized_latents_idx @ self.embedding.weight

        return quantized_latents



write the linear vqvae

In [ ]:
class LinearVectorQuantizedVAE(nn.Module):

    def __init__(self, latent_dim=2, codebook_size=512):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(32*32, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, latent_dim)
        )

        # the new layers added in from original vae model
        self.vq = VectorQuantizer(codebook_size, latent_dim)

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, 32*32),
            nn.Sigmoid()
        )


    def forward_enc(self, x):
        z = self.encoder(x)
        return z


    def quantize(self, z):
        codes = self.vq(z)

        # codebook/commitment loss
        codebook_loss = torch.mean((codes - z.detach()) ** 2)
        commitment_loss = torch.mean((codes.detach() - z) ** 2)

        codes = z + (codes - z).detach()

        return codes, codebook_loss, commitment_loss


    def forward_dec(self, x):

        codes, codebook_loss, commitment_loss = self.quantize(x)
        decoded = self.decoder(codes)

        return codes, decoded, codebook_loss, commitment_loss


    def forward(self, x):

        batch, channels, height, width = x.shape

        x = x.flatten(1)

        latents = self.forward_enc(x)
        quantized_latents, decoded, codebook_loss, commitment_loss = self.forward_dec(latents)

        decoded = decoded.reshape(batch, channels, height, width)


        return latents, quantized_latents, decoded, codebook_loss, commitment_loss

In [ ]:
data_variance = torch.var(train_set.data/255.)

In [ ]:
def VAELoss(x, x_hat, mean, log_var, kl_weight=1, reconstruction_weight=1):

    pixel_mse = ((x - x_hat)**2)

    pixel_mse = pixel_mse.flatten(1)

    reconstruction_loss = pixel_mse.sum(axis=-1).mean()

    kl = (1 + log_var - mean**2 - torch.exp(log_var))
    kl_per_image = -0.5 * torch.sum(kl, dim=-1)
    kl_loss = torch.mean(kl_per_image)

    return reconstruction_loss * reconstruction_weight + kl_loss * kl_weight

In [ ]:
def train(model,
          kl_weight,
          train_set,
          test_set,
          batch_size,
          training_iterations,
          evaluation_iterations,
          model_type="VQVAE"):

    if model_type != "VAE": kl_weight = None

    device = "cuda" if torch.cuda.is_available() else "cpu"

    model = model.to(device)

    trainloader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=8)
    testloader = DataLoader(test_set, batch_size=batch_size, shuffle=False, num_workers=8)


    optimizer = optim.Adam(model.parameters(), lr=0.0005)

    train_loss = []
    evaluation_loss = []

    encoded_data_per_eval = []
    quantized_encoded_data_per_eval = [] if model_type == "VQVAE" else None
    train_losses = []
    evaluation_losses = []

    pbar = tqdm(range(training_iterations))

    train = True

    step_counter = 0
    while train:

        for images, labels in trainloader:

            images = images.to(device)

            if model_type == "VQVAE":
                encoded, quantized_encoded, decoded, codebook_loss, commitment_loss = model(images)
                reconstruction_loss = torch.mean((images-decoded)**2) / data_variance
                loss = reconstruction_loss + codebook_loss + 0.25 * commitment_loss

            elif model_type == "VAE":
                encoded, decoded, mu, logvar = model(images)
                loss = VAELoss(images, decoded, mu, logvar, kl_weight)

            elif model_type == "AE":
                encoded, decoded = model(images)
                loss = torch.mean((images-decoded)**2)


            train_loss.append(loss.item())

            loss.backward()
            optimizer.step()
            optimizer.zero_grad()


            if step_counter % evaluation_iterations == 0:

                model.eval()

                encoded_evaluations = []

                if model_type == "VQVAE":
                    quantized_encoded_evaluations = []

                for images, labels in testloader:

                    images = images.to(device)

                    if model_type == "VQVAE":
                        encoded, quantized_encoded, decoded, codebook_loss, commitment_loss = model(images)
                        reconstruction_loss = torch.mean((images - decoded) ** 2) / data_variance
                        loss = reconstruction_loss + codebook_loss + 0.25 * commitment_loss

                    elif model_type == "VAE":
                        encoded, decoded, mu, logvar = model(images)
                        loss = VAELoss(images, decoded, mu, logvar, kl_weight)

                    elif model_type == "AE":
                        encoded, decoded = model(images)
                        loss = torch.mean((images-decoded)**2)

                    evaluation_loss.append(loss.item())

                    encoded, labels = encoded.cpu(), labels.reshape(-1, 1)

                    encoded_evaluations.append(torch.cat((encoded.flatten(1), labels), axis=-1))

                    if model_type == "VQVAE":
                        quantized_encoded = quantized_encoded.cpu()
                        quantized_encoded_evaluations.append(torch.cat((quantized_encoded.flatten(1), labels), axis=-1))


                encoded_data_per_eval.append(torch.concatenate(encoded_evaluations).detach())

                if model_type == "VQVAE":
                    quantized_encoded_data_per_eval.append(torch.concatenate(quantized_encoded_evaluations).detach())


                train_loss = np.mean(train_loss)
                evaluation_loss = np.mean(evaluation_loss)

                train_losses.append(train_loss)
                evaluation_losses.append(evaluation_loss)


                train_loss = []
                evaluation_loss = []


                model.train()

            step_counter += 1
            pbar.update(1)


            if step_counter >= training_iterations:
                print("completed training!")
                train=False
                break

    encoded_data_per_eval = [np.array(i) for i in encoded_data_per_eval]


    if model_type == "VQVAE":
        quantized_encoded_data_per_eval = [np.array(i) for i in quantized_encoded_data_per_eval]

    print("final training loss", train_losses[-1])
    print("final evaluation loss", evaluation_losses[-1])


    return model, train_losses, evaluation_losses, encoded_data_per_eval, quantized_encoded_data_per_eval



In [ ]:
class LinearVanillaAutoEncoder(nn.Module):
    def __init__(self, bottleneck_size=2):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(32*32, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, bottleneck_size)
        )

        self.decoder = nn.Sequential(
            nn.Linear(bottleneck_size, 32),
            nn.ReLU(),
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, 32*32),
            nn.Sigmoid()
        )

    def forward(self, x):

        batch, channels, height, width = x.shape

        ### Flatten Image to Vector ###
        x = x.flatten(1)

        ### Pass Through Encoder ###
        enc = self.encoder(x)

        ### Pass Through Decoder ###
        dec = self.decoder(enc)

        ### Put Decoded Image Back to Original Shape ###
        dec = dec.reshape(batch, channels, height, width)

        return enc, dec


In [ ]:
class LinearVariationalAutoEncoder(nn.Module):
    def __init__(self, latent_dim=2):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(32*32, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
        )

        #########################################################
        ### The New Layers Added in from Original AutoEncoder ###
        self.fn_mu =  nn.Linear(32, latent_dim)
        self.fn_logvar = nn.Linear(32, latent_dim)
        #########################################################

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, 32*32),
            nn.Sigmoid()
        )


    def forward_enc(self, x):

        x = self.encoder(x)

        #############################################
        ### Compute Mu and Sigma ###
        mu = self.fn_mu(x)
        logvar = self.fn_logvar(x)

        ### Sample with Reparamaterization Trick ###
        sigma = torch.exp(0.5*logvar)
        noise = torch.randn_like(sigma, device=sigma.device)
        z = mu + sigma*noise
        ############################################

        return z, mu, logvar

    def forward_dec(self, x):
        return self.decoder(x)

    def forward(self, x):

        batch, channels, height, width = x.shape

        ### Flatten Image to Vector ###
        x = x.flatten(1)

        ### Pass Through Encoder ###
        z, mu, logvar = self.forward_enc(x)

        ### Pass Sampled Data Through Decoder ###
        dec = self.decoder(z)

        ### Put Decoded Image Back to Original Shape ###
        dec = dec.reshape(batch, channels, height, width)

        return z, dec, mu, logvar

rand = torch.randn(4,1,32,32)
model = LinearVariationalAutoEncoder()
encoded, decoded, mu, logvar = model(rand)

In [ ]:
# lets train all of our linear models

linear_ae = LinearVanillaAutoEncoder()
linear_ae, train_losses, evaluation_losses, linear_ae_encoded_data_per_eval, _ = train(linear_ae,
                                                                                       kl_weight=None,
                                                                                       train_set=train_set,
                                                                                       test_set=test_set,
                                                                                       batch_size=64,
                                                                                       training_iterations=25000,
                                                                                       evaluation_iterations=250,
                                                                                       model_type="AE")

linear_vae = LinearVariationalAutoEncoder()
linear_vae, train_losses, evaluation_losses, linear_vae_encoded_data_per_eval, _ = train(linear_vae,
                                                                                         kl_weight=1,
                                                                                         train_set=train_set,
                                                                                         test_set=test_set,
                                                                                         batch_size=64,
                                                                                         training_iterations=25000,
                                                                                         evaluation_iterations=250,
                                                                                         model_type="VAE")

linear_vqvae = LinearVectorQuantizedVAE()
(linear_vqvae, train_losses,
 evaluation_losses, linear_vqvae_encoded_data_per_eval,
 linear_vqvae_quantized_encoded_data_per_eval) = train(linear_vqvae,
                                                       kl_weight=None,
                                                       train_set=train_set,
                                                       test_set=test_set,
                                                       batch_size=64,
                                                       training_iterations=25000,
                                                       evaluation_iterations=250,
                                                       model_type="VQVAE")

lets compare ae to vae to vqvae

In [ ]:
### Grab Encodings ###
encoded_latent = linear_vqvae_encoded_data_per_eval[-1]
quantized_encoded_latent = linear_vqvae_quantized_encoded_data_per_eval[-1]


fig, ax = plt.subplots(1,2, figsize=(15,5))


### Plot Continuous Encoded ###
encoded_latent = pd.DataFrame(encoded_latent, columns=["x", "y", "class"])
encoded_latent = encoded_latent.sort_values(by="class")
encoded_latent["class"] = encoded_latent["class"].astype(int).astype(str)
for grouper, group in encoded_latent.groupby("class"):
    ax[0].scatter(x=group["x"], y=group["y"], label=grouper, alpha=0.8, s=5)
ax[0].set_title("VQVAE Encoder")


### Plot Quantized ###
quantized_encoded_latent = pd.DataFrame(quantized_encoded_latent, columns=["x", "y", "class"])
quantized_encoded_latent = quantized_encoded_latent.sort_values(by="class")
quantized_encoded_latent["class"] = quantized_encoded_latent["class"].astype(int).astype(str)
for grouper, group in quantized_encoded_latent.groupby("class"):
    ax[1].scatter(x=group["x"], y=group["y"], label=grouper, alpha=0.8, s=5)
ax[1].set_title("VQVAE Quantized Encoder")
plt.show()

In [ ]:
class ConvolutionalVectorQuantizedVAE(nn.Module):

    def __init__(self, in_channels=1, latent_dim=4, codebook_size=512):
        super().__init__()

        self.bottleneck = latent_dim
        self.in_channels = in_channels
        self.codebook_size = codebook_size

        self.encoder_conv = nn.Sequential(

            # convolutional encoding
            nn.Conv2d(in_channels=in_channels, out_channels=8, kernel_size=5, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(8),
            nn.ReLU(),

            nn.Conv2d(in_channels=8, out_channels=16, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(),

            nn.Conv2d(in_channels=16, out_channels=self.bottleneck, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(self.bottleneck),
            nn.ReLU(),
        )


        # the new layers added in from original vae model
        self.vq = VectorQuantizer(codebook_size, latent_dim)


        self.decoder_conv = nn.Sequential(
            nn.ConvTranspose2d(in_channels=self.bottleneck, out_channels=16, kernel_size=3, stride=2, padding=1, output_padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(),

            nn.ConvTranspose2d(in_channels=16, out_channels=8, kernel_size=3, stride=2, padding=1, output_padding=1, bias=False),
            nn.BatchNorm2d(8),
            nn.ReLU(),

            nn.ConvTranspose2d(in_channels=8, out_channels=in_channels, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.Sigmoid()
        )


    def forward_enc(self, x):
        conv_enc = self.encoder_conv(x)
        return conv_enc


    def quantize(self, z):
        codes = self.vq(z)

        # codebook/commitment loss
        codebook_loss = torch.mean((codes - z.detach()) ** 2)
        commitment_loss = torch.mean((codes.detach() - z) ** 2)

        codes = z + (codes - z).detach()

        return codes, codebook_loss, commitment_loss


    def forward_dec(self, x):
        batch_size, channels, height, width = x.shape
        x = x.permute(0, 2, 3, 1)
        x = torch.flatten(x, start_dim=0, end_dim=-2)

        codes, codebook_loss, commitment_loss = self.quantize(x)

        codes = codes.reshape(batch_size, height, width, channels).permute(0, 3, 1, 2)
        conv_dec = self.decoder_conv(codes)

        return codes, conv_dec, codebook_loss, commitment_loss


    def forward(self, x):
        latents = self.forward_enc(x)
        codes, conv_dec, codebook_loss, commitment_loss = self.forward_dec(latents)
        return latents, codes, conv_dec, codebook_loss, commitment_loss
